# 02 · Variables y tipos de datos

**Curso de Python · Bloque 2: El lenguaje**

## Objetivos

Al terminar este notebook sabrás:

1. Qué es realmente una variable en Python (spoiler: no es una "caja").
2. Qué significa que Python tenga **tipado dinámico pero fuerte**.
3. Las particularidades de `int`, `float`, `bool`, `str` y `None`.
4. Convertir entre tipos sin caer en las trampas habituales.
5. Para qué sirven las anotaciones de tipo (*type hints*) y para qué no.

## Cómo trabajar con este notebook

- Haz una copia con tu nombre antes de empezar y selecciona el kernel del **`.venv`**.
- Ejecuta las celdas **en orden**. Si algo se comporta raro: *Restart* + *Run All*.
- 🧩 **Retos**: ejecuta, observa y escribe tu explicación en la celda ✍️ antes de seguir.
- 🎯 **Ejercicios**: completa el código y ejecuta la celda de comprobación.
- Dificultad: ⭐ fácil · ⭐⭐ medio · ⭐⭐⭐ para pensar un rato.
- Algunas celdas usan `try:` / `except` para **mostrar un error sin detener el notebook**. Lo veremos en el tema de excepciones; de momento léelo como *"intenta esto y, si falla, muestra el error"*.

---
## 1. Las variables son etiquetas, no cajas

En lenguajes como C, una variable es una **caja** en memoria con un tipo fijo donde se guarda un valor.

En Python, **los valores son objetos** que viven en memoria, y una variable es solo un **nombre (una etiqueta) que apunta a un objeto**. Asignar (`=`) no copia nada: pega la etiqueta a un objeto.

- `id(x)` devuelve la identidad del objeto (en CPython, su dirección de memoria).
- `type(x)` devuelve su tipo. **El tipo pertenece al objeto, no a la variable.**
- `a is b` comprueba si dos nombres apuntan **al mismo objeto**; `a == b` compara si tienen **el mismo valor**.

In [ ]:
x = 42
print(type(x), id(x))

x = "ahora soy texto"   # la etiqueta x se mueve a otro objeto
print(type(x), id(x))

Asignación múltiple, intercambio y desempaquetado:

In [ ]:
a, b, c = 1, 2, 3
print(a, b, c)

a, b = b, a            # intercambio sin variable temporal
print(a, b)

x = y = 0              # dos etiquetas al mismo objeto
print(x is y)

`del` **borra el nombre**, no el objeto. El objeto se libera cuando ya no queda ninguna etiqueta apuntándole.

In [ ]:
z = 100
del z
try:
    print(z)
except NameError as e:
    print(f"NameError: {e}")

### 🧩 Reto 1.1 · Dos etiquetas, un objeto  ⭐⭐

Hacemos `b = a` y después modificamos `b`. ¿Cambia `a`? ¿Qué pasa con los `id` antes y después del `+=`?

In [1]:
a = 10
b = a
print(id(a), id(b))

b += 1
print(id(a), id(b))
print("a:", a, "| b:", b)

4344469600 4344469600
4344469600 4344469632
a: 10 | b: 11


✍️ **Tu respuesta:**

Al principio `a` y `b` tienen el mismo `id`, apuntan al mismo 10. Con `b += 1` no se cambia el 10 (los int son inmutables), se crea un 11 y `b` pasa a apuntar ahí. Por eso cambia el `id` de `b` y `a` sigue valiendo 10.

### 🧩 Reto 1.2 · `is` frente a `==` (para nota)  ⭐⭐⭐

Ejecuta las tres celdas. Los valores son siempre iguales (`==` daría `True`), pero `is` cambia de resultado. ¿Por qué?

Pista 1: CPython guarda una caché de enteros pequeños.
Pista 2: piensa en **cuándo** se crea cada objeto `257` y en qué "unidad" se compila el código.

In [2]:
a = int("256")
b = int("256")
print("256 ->", a is b)

c = int("257")
d = int("257")
print("257 ->", c is d)

256 -> True
257 -> False


In [3]:
e, f = 257, 257
print("misma instrucción ->", e is f)

g = 257
h = 257
print("instrucciones distintas ->", g is h)

misma instrucción -> True
instrucciones distintas -> False


In [4]:
codigo = "x = 257\ny = 257\nprint('como script .py ->', x is y)"
exec(codigo)

como script .py -> True


✍️ **Tu respuesta:**

Los enteros de -5 a 256 están en caché, así que 256 siempre es el mismo objeto y `is` da `True`. El 257 no, cada `int("257")` crea uno nuevo.

En la misma línea Python compila todo junto y reutiliza la misma constante 257, por eso da `True`. En líneas separadas del notebook cada una se compila aparte y salen dos objetos distintos. Con `exec` el código entero se compila de una vez (como un `.py`) y vuelve a reutilizar la constante.

Conclusión: para comparar valores siempre `==`, el `is` depende de cosas internas.

### 🎯 Ejercicio 1.3 · Intercambio

Intercambia los valores de `izquierda` y `derecha` **en una sola línea** y sin usar variables auxiliares.

In [6]:
izquierda = "A"
derecha = "B"

izquierda, derecha = derecha, izquierda

In [7]:
# ✔️ Comprobación
assert (izquierda, derecha) == ("B", "A"), f"izquierda={izquierda!r}, derecha={derecha!r}"
print("¡Correcto! 🎉")

¡Correcto! 🎉


---
## 2. Tipado dinámico, pero fuerte

Son dos conceptos distintos que se confunden a menudo:

| | Significa | Python |
| --- | --- | --- |
| **Dinámico** (vs. estático) | El tipo se comprueba al **ejecutar**, no al compilar. Una variable puede apuntar a objetos de distinto tipo. | ✅ Dinámico |
| **Fuerte** (vs. débil) | El lenguaje **no convierte tipos implícitamente** cuando la operación no tiene sentido. | ✅ Fuerte |

Comparativa: Java es estático y fuerte; JavaScript es dinámico y débil; C es estático y bastante débil.

In [ ]:
valor = 10
print(type(valor))
valor = 3.5
print(type(valor))
valor = "texto"
print(type(valor))

In [ ]:
print(isinstance(10, int))
print(isinstance(10, (int, float)))   # ¿es alguno de estos tipos?
print(type(10) == int)

### 🧩 Reto 2.1 · Python frente a JavaScript  ⭐⭐

En JavaScript, `"3" + 3` da `"33"` y `"3" * 3` da `9`. ¿Qué hace Python en cada caso? ¿Por qué el segundo **no** da error?

In [8]:
try:
    print("3" + 3)
except TypeError as e:
    print(f"TypeError: {e}")

print("3" * 3)
print("3" + str(3))
print(int("3") + 3)

TypeError: can only concatenate str (not "int") to str
333
33
6


✍️ **Tu respuesta:**

`"3" + 3` da `TypeError`, Python no convierte solo el número a texto (tipado fuerte). `"3" * 3` sí funciona porque multiplicar un string por un entero es repetirlo, sale `"333"`, no 9. Si quieres sumar tienes que convertir tú con `str()` o `int()`.

### 🧩 Reto 2.2 · `isinstance` frente a `type`  ⭐⭐

¿Por qué `isinstance(True, int)` es `True` pero `type(True) == int` es `False`? ¿Cuál deberías usar normalmente?

In [9]:
print(isinstance(True, int))
print(type(True) == int)
print(type(True))
print(bool.__mro__)

True
False
<class 'bool'>
(<class 'bool'>, <class 'int'>, <class 'object'>)


✍️ **Tu respuesta:**

`bool` es una subclase de `int` (se ve en el `__mro__`). `isinstance` tiene en cuenta la herencia, así que `True` cuenta como int. `type()` mira el tipo exacto, que es `bool`, no `int`.

Normalmente mejor `isinstance`.

---
## 3. Números enteros: `int`

- **Precisión arbitraria**: no hay desbordamiento, el tamaño solo está limitado por la memoria.
- Se pueden escribir con `_` como separador visual: `1_000_000`.
- Literales en otras bases: binario `0b`, octal `0o`, hexadecimal `0x`.

In [ ]:
print(2 ** 100)
print(1_000_000 + 1)
print(0b1010, 0o17, 0xFF)
print(bin(10), oct(15), hex(255))
print(int("ff", 16), int("1010", 2))

### 🧩 Reto 3.1 · ¿Y el desbordamiento?  ⭐⭐

En C o Java, un `int` de 64 bits tiene un máximo de 9.223.372.036.854.775.807. ¿Qué pasa en Python si nos pasamos? ¿Tiene algún coste?

In [ ]:
import sys

maximo_64 = 2 ** 63 - 1
print(maximo_64)
print(maximo_64 + 1)

print(sys.getsizeof(1), "bytes")
print(sys.getsizeof(2 ** 63), "bytes")
print(sys.getsizeof(2 ** 1000), "bytes")

✍️ **Tu respuesta:**

No se desborda, sigue sumando sin problema porque los int de Python no tienen tamaño fijo. El coste es la memoria y la velocidad: cuanto más grande el número más bytes ocupa (28, 36, 160...) y más lentas son las operaciones.

### 🧩 Reto 3.2 · Un número demasiado grande para imprimirlo  ⭐⭐⭐

Python puede calcular `10 ** 5000` sin problema… pero no puede convertirlo a texto. ¿Por qué existe este límite? ¿Qué tiene que ver con la seguridad?

In [ ]:
grande = 10 ** 5000
print(grande > 0)

try:
    print(str(grande))
except ValueError as e:
    print(f"ValueError: {e}")

✍️ **Tu respuesta:**

Pasar un número tan grande a texto es muy lento (crece más que lineal). Si un programa convierte números que vienen de fuera, alguien podría mandar uno enorme y dejar el servidor colgado (DoS). Por eso desde 3.11 hay un límite de 4300 dígitos, que se puede cambiar con `sys.set_int_max_str_digits()`.

---
## 4. Números decimales: `float`

- Siguen el estándar **IEEE 754** de doble precisión (64 bits): unos 15–17 dígitos significativos.
- Admiten notación científica: `1.5e3`.
- Valores especiales: infinito (`inf`) y "no es un número" (`nan`).

In [ ]:
import sys, math

print(1.5e3, 2.5e-4)
print(sys.float_info.max)
print(sys.float_info.epsilon)

print(float("inf"), -math.inf)
print(1e308 * 10)
print(math.inf - math.inf)

### 🧩 Reto 4.1 · Redondeo bancario  ⭐⭐

¿Por qué `round(2.5)` da `2` y no `3`? ¿Y por qué `round(2.675, 2)` da `2.67`? Son dos causas **distintas**.

In [ ]:
print(round(0.5), round(1.5), round(2.5), round(3.5))
print(round(2.675, 2))

✍️ **Tu respuesta:**

`round(2.5)` da 2 porque Python usa redondeo bancario: cuando está justo a la mitad redondea al par más cercano (0.5→0, 1.5→2, 2.5→2, 3.5→4). Así no se acumula error hacia arriba.

Lo de `2.675` es otra cosa: en binario no se guarda exacto, en realidad es 2.67499999... así que redondea hacia abajo.

### 🧩 Reto 4.2 · El número que no es igual a sí mismo  ⭐⭐⭐

`nan` significa *Not a Number*. Explica los dos resultados.

In [ ]:
nan = float("nan")

print(nan == nan)
print(nan is nan)

✍️ **Tu respuesta:**

Por norma (IEEE 754) `nan` nunca es igual a nada, ni a sí mismo, así que `==` da `False`. Pero `is` mira si es el mismo objeto, y la variable es la misma, así que da `True`. Para comprobar nan se usa `math.isnan()`.

### 🧩 Reto 4.3 · Enteros que se pierden  ⭐⭐⭐

¿Por qué al convertir `2**53 + 1` a `float` se "pierde" el 1? ¿Qué implicaciones tiene, por ejemplo, para IDs grandes en un JSON?

In [ ]:
n = 2 ** 53 + 1
print(n)
print(float(n))
print(int(float(n)) == n)

✍️ **Tu respuesta:**

Un float solo tiene 53 bits para la mantisa, así que a partir de 2**53 no puede guardar todos los enteros y se redondea al más cercano que sí puede (2**53). El 1 se pierde.

En JSON pasa con JavaScript, que usa float para todos los números: un ID muy grande puede llegar cambiado. Por eso muchas APIs mandan los IDs como string.

### Cuando la precisión importa: `decimal`

Para dinero y cualquier cálculo donde los decimales deban ser exactos en base 10, se usa el módulo `decimal`.

In [ ]:
from decimal import Decimal

print(0.1 + 0.1 + 0.1)
print(Decimal("0.1") + Decimal("0.1") + Decimal("0.1"))

### 🧩 Reto 4.4 · `Decimal` mal construido  ⭐⭐

Las dos líneas usan `Decimal`, pero el resultado es muy distinto. ¿Por qué? ¿Cuál es la forma correcta?

In [10]:
from decimal import Decimal

print(Decimal(0.1))
print(Decimal("0.1"))

0.1000000000000000055511151231257827021181583404541015625
0.1


✍️ **Tu respuesta:**

`Decimal(0.1)` recibe un float que ya viene con el error del binario y lo guarda tal cual, con todos esos decimales. `Decimal("0.1")` se crea desde el texto y es exacto. Lo correcto es pasarle siempre un string.

### 🎯 Ejercicio 4.5 · Ticket exacto

Calcula `total` como la suma exacta de tres artículos de **0,10 €** usando `Decimal`, de forma que la comprobación pase.

In [11]:
from decimal import Decimal

total = Decimal("0.10") + Decimal("0.10") + Decimal("0.10")
print(total)

0.30


In [12]:
# ✔️ Comprobación
assert isinstance(total, Decimal), "total debe ser un Decimal"
assert total == Decimal("0.30"), f"total = {total}"
print("¡Correcto! 🎉")

¡Correcto! 🎉


---
## 5. Booleanos: `bool` y la *truthiness*

- Solo hay dos valores: `True` y `False` (con mayúscula).
- `bool` es una **subclase de `int`**: `True` vale 1 y `False` vale 0.
- **Cualquier objeto** puede evaluarse como verdadero o falso (*truthiness*). Son **falsos**: `False`, `None`, `0`, `0.0` y `""` (y las colecciones vacías, que veremos más adelante). Todo lo demás es verdadero.

In [ ]:
print("0       ->", bool(0))
print("1       ->", bool(1))
print("-1      ->", bool(-1))
print("0.0     ->", bool(0.0))
print("''      ->", bool(""))
print("' '     ->", bool(" "))
print("'False' ->", bool("False"))
print("None    ->", bool(None))

### 🧩 Reto 5.1 · Contar con booleanos  ⭐⭐

¿Por qué funcionan estas operaciones? ¿Le encuentras alguna utilidad práctica a la tercera?

In [ ]:
print(True + True)
print(True * 10)

nota_ana, nota_luis, nota_eva = 7, 4, 9
aprobados = (nota_ana >= 5) + (nota_luis >= 5) + (nota_eva >= 5)
print(aprobados)

✍️ **Tu respuesta:**

Funciona porque `True` vale 1 y `False` 0 (`bool` es subclase de `int`). La tercera sirve para contar cuántas condiciones se cumplen, en este caso cuántos han aprobado, sin tener que usar ifs.

---
## 6. La ausencia de valor: `None`

- `None` representa "no hay valor". Es el único objeto de tipo `NoneType`.
- Es lo que devuelve una función sin `return`.
- Es un **singleton**: solo existe un `None` en todo el programa. Por eso se compara con **`is`**.

In [ ]:
resultado = None
print(type(resultado))
print(resultado is None)
print(resultado is not None)

### 🧩 Pregunta 6.1 · ¿`== None` o `is None`?  ⭐⭐

Las dos líneas dan el mismo resultado. Entonces, ¿por qué PEP 8 dice que **siempre** hay que usar `is None` y no `== None`?

Pista: recuerda la diferencia entre `==` (compara valores) e `is` (compara identidad), y piensa en quién decide qué significa "ser igual".

In [ ]:
x = None
print(x == None)
print(x is None)

✍️ **Tu respuesta:**

`==` llama al método `__eq__` del objeto, y cualquier clase puede cambiarlo para que diga que es igual a `None` aunque no lo sea. `is` comprueba si es el mismo objeto y no se puede trucar. Como solo hay un `None`, `is None` es lo seguro (y además más rápido).

---
## 7. Cadenas de texto: `str`

- Secuencias **inmutables** de caracteres **Unicode**.
- Se pueden escribir con comillas simples o dobles (son equivalentes), o triples para varias líneas.
- Se accede por índice (`s[0]`, `s[-1]`) y por rangos (`s[1:4]`). Los métodos de cadena los veremos a fondo más adelante.

In [ ]:
s = "Python"
print(len(s), s[0], s[-1], s[0:3])
print(s.upper(), s.lower(), s.replace("P", "J"))

texto = '''Línea 1
Línea 2'''
print(texto)

ruta = r"C:\nuevos\temp"     # cadena "raw": las \ no son escapes
print(ruta)

### 🧩 Reto 7.1 · Cadenas inmutables  ⭐

¿Por qué no se puede cambiar una letra de una cadena? Entonces, ¿cómo es posible que `s.upper()` funcione? Pista: fíjate en los `id`.

In [13]:
s = "hola"
try:
    s[0] = "H"
except TypeError as e:
    print(f"TypeError: {e}")

print(id(s))
s = s.upper()
print(s, id(s))

TypeError: 'str' object does not support item assignment
4449479456
HOLA 4449482288


✍️ **Tu respuesta:**

Los strings son inmutables, no se pueden cambiar una vez creados, por eso `s[0] = "H"` da error. `upper()` no cambia `s`, crea un string nuevo y se lo asignamos otra vez a `s`. Se ve en que el `id` cambia.

### 🧩 Reto 7.2 · ¿Cuántas letras tiene?  ⭐⭐⭐

Las dos primeras cadenas se ven exactamente igual: `café`. ¿Por qué tienen distinta longitud? ¿Y por qué el emoji de la familia mide 5?

In [ ]:
cafe_1 = "café"
cafe_2 = "cafe\u0301"
familia = "👨‍👩‍👧"

print(cafe_1, cafe_2, cafe_1 == cafe_2)
print(len(cafe_1), len(cafe_2))
print(len(familia))
print(hex(ord(familia[0])), hex(ord(familia[1])), hex(ord(familia[2])), hex(ord(familia[3])), hex(ord(familia[4])))

In [ ]:
import unicodedata

print(unicodedata.normalize("NFC", cafe_2) == cafe_1)
print(len(unicodedata.normalize("NFC", cafe_2)))

✍️ **Tu respuesta:**

`cafe_1` tiene la é como un solo carácter. `cafe_2` tiene una `e` normal más un acento que se combina (`\u0301`), por eso se ve igual pero mide 5 y no son `==`. Con `unicodedata.normalize("NFC", ...)` se juntan y ya son iguales.

El emoji de familia son 3 emojis (hombre, mujer, niña) unidos por 2 caracteres invisibles (`0x200d`, el ZWJ). `len` cuenta puntos de código, no lo que se ve, así que da 5.

---
## 8. Conversión de tipos

Las funciones `int()`, `float()`, `str()` y `bool()` crean un objeto nuevo del tipo indicado a partir de otro valor. Es imprescindible cuando los datos llegan como **texto**: `input()`, ficheros CSV, formularios, variables de entorno…

In [ ]:
print(int("42") + 1)
print(float("3.14") * 2)
print(str(100) + " euros")
print(int(3.99))
print(float(7))
print(int("  42\n"))      # ignora espacios y saltos de línea alrededor

### 🧩 Reto 8.1 · Conversiones que fallan… y una que no debería funcionar  ⭐⭐

Predice el resultado de cada línea **antes** de ejecutar. ¿Cuál te ha sorprendido más?

In [14]:
print("int(float('3.7')) ->", int(float("3.7")))
print("int(-3.7)         ->", int(-3.7))
print("int('1_000')      ->", int("1_000"))
print("float('1e3')      ->", float("1e3"))

try:
    print(int("3.7"))
except ValueError as e:
    print(f"int('3.7')        -> ValueError: {e}")

try:
    print(int(""))
except ValueError as e:
    print(f"int('')           -> ValueError: {e}")

int(float('3.7')) -> 3
int(-3.7)         -> -3
int('1_000')      -> 1000
float('1e3')      -> 1000.0
int('3.7')        -> ValueError: invalid literal for int() with base 10: '3.7'
int('')           -> ValueError: invalid literal for int() with base 10: ''


✍️ **Tu respuesta:**

- `int(float('3.7'))` → 3, corta los decimales.
- `int(-3.7)` → -3, no -4: `int` trunca hacia el cero, no redondea hacia abajo.
- `int('1_000')` → 1000, acepta los `_` igual que en el código.
- `float('1e3')` → 1000.0.
- `int('3.7')` falla, `int` no acepta texto con decimales.
- `int('')` falla porque está vacío.

Lo que más me ha sorprendido es que `int('1_000')` funcione y que `int('3.7')` no.

### 🧩 Reto 8.2 · La trampa del booleano  ⭐⭐

Leemos de un fichero de configuración el valor `"False"` y lo convertimos con `bool()`. ¿Por qué el programa cree que la opción está activada?

In [15]:
valor_leido = "False"        # viene de un fichero, una variable de entorno...
modo_debug = bool(valor_leido)

print("¿Modo debug activado?", modo_debug)

¿Modo debug activado? True


✍️ **Tu respuesta:**

`bool()` no lee el texto, solo mira si el string está vacío o no. `"False"` tiene letras, así que es `True`. Hay que comparar el texto, por ejemplo `valor_leido == "True"` o `valor_leido.lower() in ("true", "1", "si")`.

### 🎯 Ejercicio 8.3 · Datos leídos de un fichero

Hemos leído una línea de un fichero y ya la tenemos separada en cuatro trozos. **Todos llegan como texto**, que es lo que ocurre siempre con ficheros CSV, formularios o variables de entorno.

Crea las variables `nombre` (str), `edad` (int), `altura` (float) y `activo` (bool) con los valores correctos. ¡Cuidado con `activo`!

In [ ]:
nombre_txt, edad_txt, altura_txt, activo_txt = "Ana", "34", "1.72", "False"

nombre = nombre_txt
edad = int(edad_txt)
altura = float(altura_txt)
activo = activo_txt == "True"

In [ ]:
# ✔️ Comprobación
assert nombre == "Ana" and type(nombre) is str, f"nombre: {nombre!r}"
assert edad == 34 and type(edad) is int, f"edad: {edad!r}"
assert altura == 1.72 and type(altura) is float, f"altura: {altura!r}"
assert activo is False, f"activo: {activo!r} (¿has usado bool()?)"
print("¡Correcto! 🎉")

### 🎯 Ejercicio 8.4 · Precio en formato español

Un usuario introduce el precio en formato español: `"1.234,56"` (punto para miles, coma para decimales). Conviértelo al `float` `1234.56`.

In [16]:
precio_texto = "1.234,56"

precio = float(precio_texto.replace(".", "").replace(",", "."))
print(precio)

1234.56


In [17]:
# ✔️ Comprobación
assert isinstance(precio, float), f"precio debe ser float, es {type(precio).__name__}"
assert precio == 1234.56, f"precio = {precio}"
print("¡Correcto! 🎉")

¡Correcto! 🎉


🧠 **Pregunta extra:** ¿qué pasa si haces los dos `replace` en el orden contrario? ¿Por qué el orden importa?

✍️ **Tu respuesta:**

Si cambias primero la coma por punto queda `"1.234.56"` y luego al quitar los puntos queda `"123456"`, o sea 123456.0, que está mal. Primero hay que quitar el punto de los miles y luego cambiar la coma por punto.

---
## 9. Anotaciones de tipo (*type hints*)

Desde Python 3.5 se puede **indicar** el tipo esperado de variables, parámetros y valores de retorno. Sirven para:

- **Documentar** el código.
- Donde más se usan es en las **funciones** (parámetros y valor devuelto); lo veremos en el tema 07.
- Que el editor (Pylance en VS Code) ofrezca mejor autocompletado y **avise de errores** antes de ejecutar.
- Herramientas de comprobación estática como **mypy** o **pyright**.

In [ ]:
nombre: str = "Ana"
edad: int = 34
altura: float = 1.72
telefono: str | None = None      # puede ser texto o None

print(nombre, edad, altura, telefono)

### 🧩 Reto 9.1 · Anotaciones que no obligan  ⭐⭐

Hemos anotado `edad` como `int` y le hemos asignado un texto. ¿Por qué Python no da error? Entonces, ¿para qué sirven las anotaciones?

Abre esta celda en VS Code: ¿te avisa el editor de algo?

In [ ]:
edad: int = "treinta y cuatro"
print(edad, type(edad))

✍️ **Tu respuesta:**

Python no comprueba las anotaciones al ejecutar, solo son información. Sirven para documentar y para que el editor (Pylance) o mypy avisen del error antes de ejecutar. En VS Code sí sale subrayado avisando de que un `str` no es `int`.

---
## ✅ Resumen

- Una variable es una **etiqueta** que apunta a un objeto. **La asignación nunca copia.**
- El tipo es del **objeto**, no de la variable. Python es de tipado **dinámico pero fuerte**.
- Los tipos vistos (`int`, `float`, `bool`, `str`, `None`) son **inmutables**: cualquier "cambio" crea un objeto nuevo.
- `==` compara **valores**; `is` compara **identidad**. Usa `is` solo con `None`.
- `int`: precisión arbitraria. `float`: IEEE 754, nunca compares con `==` (usa `math.isclose`). Dinero: `Decimal` construido desde cadena.
- `bool` es subclase de `int`. *Truthiness*: vacío/cero/`None` es falso.
- `len()` cuenta puntos de código Unicode, no letras visibles.
- Conversiones: `int()` trunca hacia cero, `bool("False")` es `True`. Los datos externos siempre llegan como texto.
- Las anotaciones de tipo documentan y ayudan al editor, pero **no se comprueban al ejecutar**.

**Siguiente notebook:** `03_operadores.ipynb`